# Trial Eligbility

**Assess number of patients in the mCRC analytic cohort meeting DreamSeq ECOG and lab eligbility.**

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorMelanoma, merge_dataframes

## Import dfs

In [2]:
index_df = pd.read_csv('../outputs/ioio_tki_index.csv')

In [3]:
index_df.shape

(2771, 3)

In [4]:
dtype_map = pd.read_csv('../outputs/ioio_tki_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/ioio_tki_features_df.csv', dtype = dtype_map)

In [5]:
features_df.shape

(1339, 174)

In [6]:
main_df = pd.merge(features_df, index_df, on = 'PatientID', how = 'left')

In [7]:
main_df = main_df.query('adv_diagnosis_year <= 2021')

In [8]:
main_df.shape

(1135, 176)

In [9]:
main_df = main_df[['PatientID', 'LineName', 'StartDate']]

In [10]:
main_df.head(3)

,PatientID,LineName,StartDate
0,F744F618949B5,ioio,2020-04-24
1,F4AAE7EB8AE49,tki,2021-03-26
2,F702FE1F825B7,tki,2020-01-06


## ECOG and Labs

In [11]:
# Initialize class 
processor = DataProcessorMelanoma()

In [12]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = main_df,
                                 index_date_column = 'StartDate',
                                 additional_loinc_mappings = {'neutrophil': ['26499-4', '751-8', '753-4']},
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-04-25 23:13:32,902 - INFO - Successfully read Lab.csv file with shape: (10819406, 17) and unique PatientIDs: 16150
2026-04-25 23:13:34,896 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (996880, 18) and unique PatientIDs: 1115
2026-04-25 23:13:37,092 - INFO - Successfully processed Lab.csv file with final shape: (1135, 86) and unique PatientIDs: 1135


In [13]:
labs_df = labs_df[['PatientID', 'wbc', 'neutrophil', 'hemoglobin', 'platelet', 'creatinine', 'total_bilirubin', 'ast', 'alt', 'alp']]

In [14]:
labs_df.shape

(1135, 10)

In [15]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = main_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-04-25 23:13:37,203 - INFO - Successfully read ECOG.csv file with shape: (211808, 4) and unique PatientIDs: 12139
2026-04-25 23:13:37,233 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (20929, 5) and unique PatientIDs: 911
2026-04-25 23:13:37,250 - INFO - Successfully processed ECOG.csv file with final shape: (1135, 3) and unique PatientIDs: 1135


In [16]:
ecog_df = ecog_df[['PatientID', 'ecog_index']]

In [17]:
ecog_df.shape

(1135, 2)

In [18]:
main_df = pd.merge(main_df, labs_df, on = 'PatientID', how = 'left')

In [19]:
main_df = pd.merge(main_df, ecog_df, on = 'PatientID', how = 'left')

In [20]:
main_df.shape

(1135, 13)

In [21]:
main_df.head(3)

,PatientID,LineName,StartDate,wbc,neutrophil,hemoglobin,platelet,creatinine,total_bilirubin,ast,alt,alp,ecog_index
0,F744F618949B5,ioio,2020-04-24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,F4AAE7EB8AE49,tki,2021-03-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
2,F702FE1F825B7,tki,2020-01-06,5.9,NaN,15.6,239.0,0.76,0.4,22.0,28.0,82.0,0


## ECOG Eligibility

In [22]:
main_df['ecog_eligible'] = np.where((main_df['ecog_index'] <= 1), 1, 0)

In [23]:
main_df['ecog_ineligible'] = np.where(main_df['ecog_index'] > 1, 1, 0)

In [24]:
main_df['ecog_indeterminate'] = np.where(main_df['ecog_index'].isna(), 1, 0)

In [25]:
main_df.shape[0]

1135

In [26]:
main_df.ecog_eligible.value_counts()[1]+main_df.ecog_ineligible.value_counts()[1]+main_df.ecog_indeterminate.value_counts()[1]

np.int64(1135)

## Lab eligibility

In [27]:
main_df['lab_eligible'] = np.where(
    (main_df['wbc'] >= 3) &
    (main_df['neutrophil'] >= 1.5) &
    (main_df['platelet'] >= 100) &
    (main_df['hemoglobin'] >= 8) &
    (main_df['creatinine'] <= 1.8) & 
    (main_df['total_bilirubin'] <= 1.8) &
    (main_df['ast'] <= 120) &
    (main_df['alt'] <= 120) &
    (main_df['alp'] <= 300), 1, 0)

In [28]:
main_df['lab_ineligible'] = np.where(
    (main_df['wbc'] < 3) |
    (main_df['neutrophil'] < 1.5) |
    (main_df['platelet'] < 100) |
    (main_df['hemoglobin'] < 8) |
    (main_df['creatinine'] > 1.8) | 
    (main_df['total_bilirubin'] > 1.8) |
    (main_df['ast'] > 120) |
    (main_df['alt'] > 120) |
    (main_df['alp'] > 300), 1, 0)

In [29]:
main_df['lab_indeterminate'] = np.where((main_df['lab_eligible'] == 0) & (main_df['lab_ineligible'] == 0), 1, 0)

In [30]:
main_df.lab_eligible.value_counts()[1]+main_df.lab_ineligible.value_counts()[1]+main_df.lab_indeterminate.value_counts()[1]

np.int64(1135)

## Overal eligibility

In [31]:
main_df['eligible'] = np.where((main_df['ecog_eligible'] == 1) & (main_df['lab_eligible'] == 1), 1, 0)

In [32]:
main_df['ineligible'] = np.where((main_df['ecog_ineligible'] == 1) | (main_df['lab_ineligible'] == 1), 1, 0)

In [33]:
main_df['indeterminate'] = np.where((main_df['eligible'] == 0) & (main_df['ineligible'] == 0), 1, 0)

In [34]:
print('Overall cohort')
print(f'Eligible: count {main_df.eligible.value_counts()[1]}, percentage {round(main_df.eligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Ineligible: count {main_df.ineligible.value_counts()[1]}, percentage {round(main_df.ineligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Indeterminate: count {main_df.indeterminate.value_counts()[1]}, percentage {round(main_df.indeterminate.value_counts(normalize = True)[1]*100, 2)}')

Overall cohort
Eligible: count 411, percentage 36.21
Ineligible: count 185, percentage 16.3
Indeterminate: count 539, percentage 47.49
